In [ ]:
# Following installed mlflow version 3.10.1

# !pip install mlflow

In [1]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Set experiment
mlflow.set_experiment("Model_Compare_RF_vs_SVM_v1")

# Load data
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y)

2026/03/30 12:19:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/30 12:19:02 INFO mlflow.store.db.utils: Updating database tables
2026/03/30 12:19:05 INFO mlflow.tracking.fluent: Experiment with name 'Model_Compare_RF_vs_SVM_v1' does not exist. Creating a new experiment.


In [3]:
# Took 10 seconds
def train_and_log(model, model_name, params):
    with mlflow.start_run(run_name=model_name):
        model.set_params(**params)
        model.fit(X_train, y_train)

        acc = model.score(X_test, y_test)

        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.set_tag("model_name", model_name)

        mlflow.sklearn.log_model(model, name="model", serialization_format="skops")

# Compare models
train_and_log(RandomForestClassifier(), "RandomForest", {"n_estimators": 100})
train_and_log(SVC(), "SVM", {"kernel": "rbf"})

print("Clean comparison done ")

2026/03/30 12:21:13 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\hi\AppData\Local\Temp\tmp689axd3b\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/30 12:21:19 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\hi\AppData\Local\Temp\tmp93g4qvc9\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 


Clean comparison done 


# Now access one of models through its API endpoints

In [6]:
import requests
import numpy as np
from sklearn.datasets import load_iris

# Load iris data
iris = load_iris()
X = iris.data

# Take one sample (or multiple)
sample = X[89:96]   # try 2 rows

# Endpoint
url = "http://127.0.0.1:5002/invocations"

# Payload format (VERY IMPORTANT)
data = {"inputs": sample.tolist()}

# Send request
response = requests.post(url, json=data)

print("Status Code:", response.status_code)
print("Prediction:", response.json())


Status Code: 200
Prediction: {'predictions': [1, 1, 1, 1, 1, 1, 1]}
